# Extensión de la Ontología de Pizzas y razonamiento semántico en pizzerías

El objetivo de esta práctica es extender una Ontología estándar de pizzas en OWL para dar soporte al modelo de negocio de una pizzería.

El fichero con la Ontología, llamado `pizza.owl`, lo puedes encontrar en PoliformaT.

## Descripción del problema

La Ontología base `pizza.owl` únicamente describe el dominio alimentario (tipos de masa, toppings y categorías de pizza).

Para adaptar este modelo a la gestión real de un establecimiento, es necesario incorporar el modelo transaccional y de clientes.

El objetivo principal es modelar reglas de negocio explícitas e implícitas mediante Lógica de Descripción (DL) en Owlready2, de manera que el motor de razonamiento semántico (Pellet) pueda clasificar automáticamente a los clientes dentro de programas de fidelización (como *VIP* o *Preferente*) e inferir categorías de pedidos sin necesidad de programar condicionales manuales en Python.

## Estructura del trabajo:
* Parte 1: Extensión del modelo (TBox) con Owlready2.
* Parte 2: Poblado de datos (ABox) con Owlready2.
* Parte 3: Razonamiento semántico, incosistencia y consultas en Owlready2.


## Instalación e importación de Owlready2

In [ ]:
%pip install owlready2

In [3]:
from owlready2 import *
import subprocess
subprocess.run(['java', '-version'])

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

## Visualización gráfica de Ontología

Con el objetivo de facilitaros la inspección del esquema conceptual (TBox), os proporcionamos un fragmento de código que analiza la Ontología y genera un diagrama visual en formato `.png`.

**Antes de ejecutarlo**: Aseguraos de haber cargado previamente la Ontología en memoria (objeto `onto`).

Al ejecutar el bloque de código, la imagen se guardará automáticamente en vuestro directorio de trabajo.


In [2]:
%pip install graphviz

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Requiere: %pip install owlready2 graphviz
import graphviz
from owlready2 import *

########   1. CARGAR la ontología   ########

onto = get_ontology('pizza.owl').load()

############################################

# 2. Configurar el grafo con orientacion UML (de abajo a arriba)
dot = graphviz.Digraph('PizzaOntology', comment='Diagrama de Clases Pizza', format='png')
dot.attr(rankdir='BT', size='50,50')

# 3. Dibujar nodos y jerarquías
for cls in onto.classes():
    # Estilo especial según la categoría principal
    if cls in [onto.Pizza, onto.PizzaBase, onto.PizzaTopping, onto.Country]:
        dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#ffcccb')
    else:
        dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#e1f5fe')

    # Subclases (subClassOf)
    for parent in cls.is_a:
        if isinstance(parent, ThingClass):
            dot.edge(cls.name, parent.name, label='subClassOf')

# 4. Exportar y renderizar
output_path = dot.render('diagrama_pizza', view=False)
print(f'Imagen generada con éxito en: {output_path}')

Imagen generada con éxito en: diagrama_pizza.png


In [15]:
import graphviz
from owlready2 import *


def visualize_owl(name_owl):
    

    ########   1. CARGAR la ontología   ########

    onto = get_ontology(name_owl).load()

    ############################################

    # 2. Configurar el grafo con orientacion UML (de abajo a arriba)
    dot = graphviz.Digraph('PizzaOntology', comment='Diagrama de Clases Pizza', format='png')
    dot.attr(rankdir='BT', size='50,50')

    # 3. Dibujar nodos y jerarquías
    for cls in onto.classes():
        # Estilo especial según la categoría principal
        if cls in [onto.Pizza, onto.PizzaBase, onto.PizzaTopping, onto.Country]:
            dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#ffcccb')
        else:
            dot.node(cls.name, shape='rectangle', style='filled', fillcolor='#e1f5fe')

        # Subclases (subClassOf)
        for parent in cls.is_a:
            if isinstance(parent, ThingClass):
                dot.edge(cls.name, parent.name, label='subClassOf')

    # 4. Exportar y renderizar
    output_path = dot.render('diagrama_pizza', view=False)
    print(f'Imagen generada con éxito en: {output_path}')

## **Parte 1**: Extensión del Modelo (TBox) con Owlready2

En esta primera parte se extenderá la estructura conceptual de la Ontología definiendo las nuevas entidades, sus relaciones y restricciones lógicas, para posteriormente crear las instancias de prueba utilizando las clases y sintaxis nativas de Owlready2. Se pide:

1. Cargar el fichero `pizza.owl`.
2. Crear dentro del contexto de la Ontología las clases base Persona, Cliente (subclase de Persona) y Pedido.
3. Definir las siguientes propiedades de objeto estableciendo explícitamente sus dominios, rangos y propiedades inversas:
    * `realizaPedido`: que implica que un cliente realiza un pedido.
    * `esRealizadoPor`: inversa de `realizaPedido`.
    * `contienePizza`: que indica que un pedido contiene una pizza.
    * `esPizzaDe`: inversa de `contienePizza`.
4. Definir las siguientes clases equivalentes:
    * `PedidoGrande`: todo `Pedido`que contiene al menos 3 pizzas.
    * `ClienteVIP`: todo `Cliente` que realiza al menos un `PedidoGrande`.
    * `ClientePreferente`: todo `Cliente` que realiza al menos 2 pedidos.

5. Declarar la disyunción explícita entre clases raíz:  `Persona`, `Pedido`, `Pizza`, `PizzaBase` y `PizzaTopping`.

Al finalizar esta parte puedes visualizar la Ontología para comprobar que se han incluido las clases base.

## Nota: sintaxis de restricciones sobre propiedades en Owlready2

**1. Restricciones de cuantificación**:
  * **`propiedad.some(Clase)` (existencial, $\exists$)**: exige que exista al menos una relación con un elemento de esa clase.
  * **`propiedad.only(Clase)` (universal, $\forall$)**: obliga a que todas las relaciones existentes sean exclusivamente de esa clase.

**2. Restricciones de cardinalidad**:
  * **`propiedad.min(N, Clase)` ($\ge N$)**: exige relacionarse con al menos $N$ instancias.
  * **`propiedad.max(N, Clase)` ($\le N$)**: limita la relación a como máximo $N$ instancias.
  * **`propiedad.exactly(N, Clase)` ($= N$)**: obliga a relacionarse con exactamente $N$ instancias.

In [8]:
onto = get_ontology('pizza.owl').load()
for cl in onto.classes():
    print(cl.name)

Pizza
PizzaBase
Country
PizzaTopping
AmericanHotPizza
NamedPizza
JalapenoPepperTopping
MozzarellaTopping
PepperoniTopping
TomatoTopping
AmericanPizza
AnchovyTopping
SeafoodTopping
CaperTopping
VegetableTopping
CheeseTopping
CheesyPizza
DeepPanBase
ThinAndCrispyBase
GreenPepperTopping
PepperTopping
HamTopping
MeatTopping
MargheritaPizza
ParmesanTopping
MushroomTopping
OliveTopping
OnionTopping
PrawnTopping
RedPepperTopping
SalamiTopping
SohoPizza
SpicyBeefTopping
TunaTopping
VegetarianPizza


In [13]:
from owlready2 import *

# 1. Cargar la ontología
onto = get_ontology("pizza.owl").load()

# 2. Crear las clases base
with onto:
    class Persona(Thing):
        pass

    class Cliente(Persona):
        pass

    class Pedido(Thing):
        pass


# 3. Crear las propiedades de objeto
with onto:
    class realizaPedido(ObjectProperty):
        domain = [Cliente]
        range = [Pedido]

    class esRealizadoPor(ObjectProperty):
        domain = [Pedido]
        range = [Cliente]

    # Declarar las propiedades inversas
    realizaPedido.inverse_property = esRealizadoPor

    class contienePizza(ObjectProperty):
        domain = [Pedido]
        range = [onto.Pizza]

    class esPizzaDe(ObjectProperty):
        domain = [onto.Pizza]
        range = [Pedido]

    contienePizza.inverse_property = esPizzaDe


# 4. Crear las clases equivalentes
with onto:
    class PedidoGrande(Thing):
        equivalent_to = [
            Pedido & contienePizza.min(3, onto.Pizza)
        ]

    class ClienteVIP(Thing):
        equivalent_to = [
            Cliente & realizaPedido.some(PedidoGrande)
        ]

    class ClientePreferente(Thing):
        equivalent_to = [
            Cliente & realizaPedido.min(2, Pedido)
        ]


# 5. Declarar las clases disjuntas
with onto:
    AllDisjoint([
        onto.Persona,
        onto.Pedido,
        onto.Pizza,
        onto.PizzaBase,
        onto.PizzaTopping
    ])


# 6. Guardar la ontología
onto.save(file="pizza_extendida.owl", format="rdfxml")

print("Ontología ampliada correctamente.")


Ontología ampliada correctamente.


In [17]:
visualize_owl('pizza_extendida.owl')

Imagen generada con éxito en: diagrama_pizza.png


## **Parte 2**: Poblado de instancias (ABox) con Owlready2

En esta segunda parte se definirán nuevas instancias específicas del dominio que se está modelando. En concreto, se pide crear las siguientes instancias sin asignar explícitamente las clases `ClienteVIP` o `ClientePreferente`, ya que deberán ser inferidas posteriormente por el razonador semántico:

1. Pizzas personalizadas:
    * `pizza_custom_1`: tiene una base `ThinAndCrispyBase` y topping `MozzarellaTopping`.
    * `pizza_custom_2`: tiene una base `ThinAndCrispyBase` y topping `OliveTopping`.
    * `pizza_custom_3`: tiene una base `ThinAndCrispyBase` y topping `ParmesanTopping`.

2. Cliente 1 (`juan_cano`):
    * Realiza un único pedido (`pedido_101`).
    * `pedido_101` contiene 3 pizzas: una `MargheritaPizza`, una `AmericanPizza` y la pizza personalizada (`pizza_custom_1`).

3. Cliente 2 (`ana_uceda`):
    * Realiza dos pedidos independientes (`pedido_102` y `pedido_103`).
    * `pedido_102` contiene 1 pizza `SohoPizza`.
    * `pedido_103` contiene 1 pizza `MargheritaPizza`.

2. Cliente 3 (`carlos_puente`):
    * Realiza un único pedido (`pedido_104`).
    * `pedido_104` contiene 3 pizzas: una `SohoPizza`, una `AmericanHotPizza` y la pizza personalizada (`pizza_custom_2`).

3. Cliente 4 (`maria_perez`):
    * Realiza dos pedidos independientes (`pedido_105` y `pedido_106`).
    * `pedido_105` contiene 1 pizza `MargheritaPizza`.
    * `pedido_106` contiene la pizza personalizada `pizza_custom_3`.



In [1]:

from owlready2 import *

# ============================================================
# PARTE 1: EXTENSION DEL MODELO (TBOX)
# ============================================================

# 1. Cargar la ontologia original
onto = get_ontology("pizza.owl").load()


# 2. Crear las clases base
with onto:

    class Persona(Thing):
        pass

    class Cliente(Persona):
        pass

    class Pedido(Thing):
        pass


# 3. Definir las propiedades de objeto
with onto:

    class realizaPedido(ObjectProperty):
        domain = [Cliente]
        range = [Pedido]

    class esRealizadoPor(ObjectProperty):
        domain = [Pedido]
        range = [Cliente]

    realizaPedido.inverse_property = esRealizadoPor

    class contienePizza(ObjectProperty):
        domain = [Pedido]
        range = [onto.Pizza]

    class esPizzaDe(ObjectProperty):
        domain = [onto.Pizza]
        range = [Pedido]

    contienePizza.inverse_property = esPizzaDe


# 4. Definir las clases equivalentes
with onto:

    class PedidoGrande(Thing):
        equivalent_to = [
            Pedido & contienePizza.min(3, onto.Pizza)
        ]

    class ClienteVIP(Thing):
        equivalent_to = [
            Cliente & realizaPedido.some(PedidoGrande)
        ]

    class ClientePreferente(Thing):
        equivalent_to = [
            Cliente & realizaPedido.min(2, Pedido)
        ]


# 5. Declarar la disyuncion explicita
with onto:
    AllDisjoint([
        Persona,
        Pedido,
        onto.Pizza,
        onto.PizzaBase,
        onto.PizzaTopping
    ])


# ============================================================
# PARTE 2: POBLADO DE INSTANCIAS (ABOX)
# ============================================================

# 6. Comprobar y crear OliveTopping si no existe
if onto["OliveTopping"] is None:
    with onto:
        class OliveTopping(onto.PizzaTopping):
            pass


# 7. Crear las pizzas personalizadas, sus bases y toppings
with onto:

    # Bases
    base_custom_1 = onto.ThinAndCrispyBase("base_custom_1")
    base_custom_2 = onto.ThinAndCrispyBase("base_custom_2")
    base_custom_3 = onto.ThinAndCrispyBase("base_custom_3")

    # Toppings
    mozzarella_custom_1 = onto.MozzarellaTopping(
        "mozzarella_custom_1"
    )
    olive_custom_2 = onto.OliveTopping(
        "olive_custom_2"
    )
    parmesan_custom_3 = onto.ParmesanTopping(
        "parmesan_custom_3"
    )

    # Pizzas personalizadas
    pizza_custom_1 = onto.Pizza("pizza_custom_1")
    pizza_custom_2 = onto.Pizza("pizza_custom_2")
    pizza_custom_3 = onto.Pizza("pizza_custom_3")

    # Asignar las bases
    # Se usa asignacion directa si hasBase es funcional.
    pizza_custom_1.hasBase = base_custom_1
    pizza_custom_2.hasBase = base_custom_2
    pizza_custom_3.hasBase = base_custom_3

    # Asignar toppings
    pizza_custom_1.hasTopping = [mozzarella_custom_1]
    pizza_custom_2.hasTopping = [olive_custom_2]
    pizza_custom_3.hasTopping = [parmesan_custom_3]


# 8. Crear los clientes y los pedidos
with onto:

    # Clientes
    juan_cano = onto.Cliente("juan_cano")
    ana_uceda = onto.Cliente("ana_uceda")
    carlos_puente = onto.Cliente("carlos_puente")
    maria_perez = onto.Cliente("maria_perez")

    # Pedidos
    pedido_101 = onto.Pedido("pedido_101")
    pedido_102 = onto.Pedido("pedido_102")
    pedido_103 = onto.Pedido("pedido_103")
    pedido_104 = onto.Pedido("pedido_104")
    pedido_105 = onto.Pedido("pedido_105")
    pedido_106 = onto.Pedido("pedido_106")


# 9. Relacionar clientes con pedidos
juan_cano.realizaPedido = [pedido_101]

ana_uceda.realizaPedido = [
    pedido_102,
    pedido_103
]

carlos_puente.realizaPedido = [pedido_104]

maria_perez.realizaPedido = [
    pedido_105,
    pedido_106
]


# 10. Crear las pizzas de los pedidos
with onto:

    # Pedido 101: Juan
    margherita_101 = onto.MargheritaPizza("margherita_101")
    american_101 = onto.AmericanPizza("american_101")

    # Pedido 102: Ana
    soho_102 = onto.SohoPizza("soho_102")

    # Pedido 103: Ana
    margherita_103 = onto.MargheritaPizza("margherita_103")

    # Pedido 104: Carlos
    soho_104 = onto.SohoPizza("soho_104")
    american_hot_104 = onto.AmericanHotPizza("american_hot_104")

    # Pedido 105: Maria
    margherita_105 = onto.MargheritaPizza("margherita_105")


# 11. Relacionar los pedidos con sus pizzas
pedido_101.contienePizza = [
    margherita_101,
    american_101,
    pizza_custom_1
]

pedido_102.contienePizza = [soho_102]

pedido_103.contienePizza = [margherita_103]

pedido_104.contienePizza = [
    soho_104,
    american_hot_104,
    pizza_custom_2
]

pedido_105.contienePizza = [margherita_105]

pedido_106.contienePizza = [pizza_custom_3]


# 12. Guardar la ontologia ampliada
onto.save(
    file="pizza_extendida.owl",
    format="rdfxml"
)

print("Ontologia ampliada y poblada correctamente.")

# ============================================================
# 13. Mostrar los individuos creados
# ============================================================

print("\nClientes:")
for cliente in [juan_cano, ana_uceda, carlos_puente, maria_perez]:
    print(cliente.name, "->", [p.name for p in cliente.realizaPedido])

print("\nPedidos y numero de pizzas:")
for pedido in [
    pedido_101, pedido_102, pedido_103,
    pedido_104, pedido_105, pedido_106
]:
    print(pedido.name, "->", len(pedido.contienePizza), "pizzas")


Ontologia ampliada y poblada correctamente.

Clientes:
juan_cano -> ['pedido_101']
ana_uceda -> ['pedido_102', 'pedido_103']
carlos_puente -> ['pedido_104']
maria_perez -> ['pedido_105', 'pedido_106']

Pedidos y numero de pizzas:
pedido_101 -> 3 pizzas
pedido_102 -> 1 pizzas
pedido_103 -> 1 pizzas
pedido_104 -> 3 pizzas
pedido_105 -> 1 pizzas
pedido_106 -> 1 pizzas


In [ ]:
# Declarar que los individuos creados son distintos entre sí.
# OWL no asume nombres únicos (UNA): sin esta declaración el razonador no puede
# contar 3 pizzas distintas (PedidoGrande) ni 2 pedidos distintos (ClientePreferente),
# porque varios nombres podrían referirse al mismo individuo.
with onto:
    AllDifferent([pizza_custom_1, pizza_custom_2, pizza_custom_3,
                  margherita_101, american_101, soho_102, margherita_103,
                  soho_104, american_hot_104, margherita_105])
    AllDifferent([pedido_101, pedido_102, pedido_103, pedido_104, pedido_105, pedido_106])
    AllDifferent([juan_cano, ana_uceda, carlos_puente, maria_perez])

print("Individuos declarados como distintos (AllDifferent).")


In [2]:
onto.save(
    file="pizza_extendida.owl",
    format="rdfxml"
)

print("Instancias creadas y ontología guardada.")

Instancias creadas y ontología guardada.


## **Parte 3**: Razonamiento semántico, inconsistencia y consultas en Owlready2

En esta tercera parte se invocará el razonador desde Owlready2 para comprobar el proceso de inferencia semántica, se realizarán consultas sobre los datos inferidos y se validará el control de inconsistencias lógicas capturando excepciones propias de Owlready2. Se piden lon siguientes 3 apartados:

**1. Invocación del razonador e inferencia**
  * Ejecutar el razonador semántico (`sync_reasoner_pellet()`) de Owlready2.
  * Imprimir por pantalla la clase inferida para `pedido_101`.
  * Imprimir por pantalla las clases inferidas para `juan_cano`.
  * Imprimir por pantalla las clases inferidas para `ana_uceda`.
  * Verificar si `pizza_custom_1` fue clasificada automáticamente como `CheesyPizza`.

Ejecuta el razonador dentro de un bloque `try-except`. Capturar específicamente la excepción `OwlReadyInconsistentOntologyError` que lanza `Owlready2` cuando detecta un modelo incoherente, mostrando un mensaje explicativo por pantalla.


In [ ]:
import owlready2.reasoning

# Configuración de Java para Pellet:
# - El Pellet incluido en owlready2 (p. ej. la versión 0.51) necesita Java 25 o superior de 64 bits.
# - Si `java` del PATH es otra versión, indica la ruta al java.exe correcto:
# owlready2.JAVA_EXE = r"C:\Program Files\Eclipse Adoptium\jdk-25...\bin\java.exe"
owlready2.reasoning.JAVA_MEMORY = 1000   # MB de memoria para la JVM

try:
    with onto:
        sync_reasoner_pellet()
    print("Razonamiento completado: la ontología es consistente.\n")
except OwlReadyInconsistentOntologyError:
    print("La ontología es INCONSISTENTE: el razonador ha encontrado hechos que "
          "contradicen los axiomas de la TBox, por lo que no se puede inferir nada.")
except OwlReadyJavaError as e:
    print("No se pudo ejecutar Pellet (problema con Java):")
    print(e)

# Clase inferida para pedido_101
print("Clases de pedido_101:", onto.pedido_101.is_a)

# Clases inferidas para juan_cano y ana_uceda
print("Clases de juan_cano:", onto.juan_cano.is_a)
print("Clases de ana_uceda:", onto.ana_uceda.is_a)

# ¿pizza_custom_1 se ha clasificado como CheesyPizza?
if isinstance(onto.pizza_custom_1, onto.CheesyPizza):
    print("pizza_custom_1 SÍ ha sido clasificada como CheesyPizza "
          "(tiene un topping MozzarellaTopping, que es un CheeseTopping).")
else:
    print("pizza_custom_1 NO ha sido clasificada como CheesyPizza.")
print("Clases de pizza_custom_1:", onto.pizza_custom_1.is_a)


**2. Consultas**
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva el nombre de todos los clientes inferidos como `ClienteVIP` junto con los identificadores de sus pedidos.
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva el nombre de todos los clientes inferidos como `ClientePreferente` junto con los identificadores de sus pedidos.
  * Escribir y ejecutar una consulta sobre la Ontología razonada que devuelva las pizzas que han sido clasificacdas como `CheesyPizza` junto con los clientes y los identificadores de pedidos en los que se pidieron.
  * Escribir y ejecutar una consulta sobre la Ontología razondad que devuelvan los identificadores de pedidos con pizzas preparadas con una masa `ThinAndCrispyBase`.

**Nota: método `onto.search()` en Olwready2 para realizar consultas**:

Para realizar consultas a la ontología desde Python, Owlready2 ofrece el método `onto.search()`. Este método explora la base de conocimiento y devuelve una lista de Python con los objetos que cumplen los criterios especificados.

Parámetros habituales de búsqueda:

* `type`: filtra entidades según la clase a la que pertenecen o de la que son instancias.
* `iri`: filtra por el nombre o identificador de la entidad (admite comodines como *).
* `Propiedades directas` (`nombre_propiedad=valor`): filtra entidades que tengan un valor o relación específica asociada a una propiedad.

Ejemplos de uso:

```
# 1. Buscar todas las instancias que pertenecen a la clase 'VegetarianPizza'
pizzas_vegetarianas = onto.search(type=onto.VegetarianPizza)

# 2. Buscar todas las pizzas que contengan un ingrediente específico (GarlicTopping)
pizzas_con_ajo = onto.search(hasTopping=onto.GarlicTopping)

# 3. Buscar cualquier entidad cuyo nombre contenga la palabra "sauce"
salsas = onto.search(iri="*sauce*")
```

Una vez obtenida la lista de resultados de `search()`, se pueden recorrer sus elementos con un bucle e inspeccionar sus propiedades o clases con la sintaxis habitual de objetos en Python (por ejemplo, `entidad.name` o `entidad.is_a`).

In [ ]:
# Consulta 1: clientes inferidos como ClienteVIP y sus pedidos
print("=== Clientes VIP ===")
for cliente in onto.search(type=onto.ClienteVIP):
    print(f"{cliente.name}: {[pedido.name for pedido in cliente.realizaPedido]}")

# Consulta 2: clientes inferidos como ClientePreferente y sus pedidos
print("\n=== Clientes Preferentes ===")
for cliente in onto.search(type=onto.ClientePreferente):
    print(f"{cliente.name}: {[pedido.name for pedido in cliente.realizaPedido]}")

# Consulta 3: pizzas clasificadas como CheesyPizza, con los pedidos y clientes en que se pidieron
print("\n=== Pizzas CheesyPizza ===")
for pizza in onto.search(type=onto.CheesyPizza):
    for pedido in pizza.esPizzaDe:
        clientes = [cliente.name for cliente in pedido.esRealizadoPor]
        print(f"{pizza.name}: pedido {pedido.name}, cliente(s) {clientes}")

# Consulta 4: pedidos con pizzas de masa ThinAndCrispyBase
print("\n=== Pedidos con pizzas de masa ThinAndCrispyBase ===")
pedidos_masa_fina = set()
for masa in onto.search(type=onto.ThinAndCrispyBase):
    for pizza in onto.search(hasBase=masa):
        for pedido in pizza.esPizzaDe:
            pedidos_masa_fina.add(pedido.name)
            print(f"{pedido.name}: {pizza.name} (masa {masa.name})")
print("Pedidos:", sorted(pedidos_masa_fina))


**3. Control de inconsistencia lógica**
  * Inconsistencia por disyunción explícita de clases:
      *   Crear una nueva instancia `individuo_inconsistente`.
      * Asignarle simultáneamente dos clases declaradas como disjuntas en la TBox (por ejemplo, `Persona` y `Pizza`).

  * Inconsistencia por violación de reglas:
      * Crear una instancia declarada explícitamente como `VegetarianPizza`.
      * Añadirle un ingrediente perteneciente a la jerarquía de carnes `PepperoniTopping` o `MeatTopping`).

Ejecuta el razonador dentro de un bloque `try-except`. Capturar específicamente la excepción `OwlReadyInconsistentOntologyError` que lanza `Owlready2` cuando detecta un modelo incoherente, mostrando un mensaje explicativo por pantalla.



In [ ]:
def comprobar_consistencia(descripcion, motivo=""):
    # Ejecuta Pellet y captura la inconsistencia si se produce
    try:
        with onto:
            sync_reasoner_pellet()
        print(f"[{descripcion}] La ontología es consistente.\n")
    except OwlReadyInconsistentOntologyError:
        print(f"[{descripcion}] Ontología INCONSISTENTE detectada por el razonador.")
        print(f"  Motivo: {motivo}\n")
    except OwlReadyJavaError as e:
        print(f"[{descripcion}] No se pudo ejecutar Pellet (problema con Java):\n{e}")


# Caso 1: inconsistencia por disyunción explícita de clases
with onto:
    individuo_inconsistente = onto.Persona("individuo_inconsistente")
    individuo_inconsistente.is_a.append(onto.Pizza)   # Persona y Pizza son disjuntas

comprobar_consistencia("Disyunción Persona / Pizza",
                       "individuo_inconsistente es a la vez Persona y Pizza, pero en la TBox se declaró "
                       "AllDisjoint([Persona, Pedido, Pizza, PizzaBase, PizzaTopping]).")

# Se elimina el individuo para que la ontología vuelva a ser consistente
destroy_entity(individuo_inconsistente)


# Caso 2: inconsistencia por violación de reglas (pizza vegetariana con carne)
with onto:
    pepperoni_1 = onto.PepperoniTopping("pepperoni_1")
    pizza_vegetariana_carne = onto.VegetarianPizza("pizza_vegetariana_carne", hasTopping=[pepperoni_1])

comprobar_consistencia("VegetarianPizza con PepperoniTopping",
                       "VegetarianPizza exige hasTopping.only(CheeseTopping | VegetableTopping), pero "
                       "PepperoniTopping es un MeatTopping, disjunto de CheeseTopping y VegetableTopping.")

destroy_entity(pizza_vegetariana_carne)
destroy_entity(pepperoni_1)


# Tras eliminar los individuos erróneos, la ontología vuelve a ser consistente
comprobar_consistencia("Ontología tras eliminar los individuos erróneos")
